# Exploring the NASA battery dataset

Before training anything, I want to understand what's actually in the data. The markdown cells under each plot are where I write down what I see — those observations end up shaping the feature engineering and the report's discussion section.

**Replace the `*Your observation here:*` blocks with what you actually notice when you run the cells. Don't skip them. They're the most useful part of the notebook for your write-up.**

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

df = pd.read_csv('../data/processed/cycles.csv')
print(df.shape)
df.head()

In [ ]:
# How many cycles per cell? Are the cells comparable in length?
df.groupby('cell_id')['cycle'].agg(['min', 'max', 'count'])

*Your observation here:* Which cell has the fewest cycles? What does that tell you about its degradation rate? Why might that matter when choosing the held-out cell?

In [ ]:
# The first plot anyone should make: SOH vs cycle, per cell.
fig, ax = plt.subplots(figsize=(8, 5))
for cell_id, cell in df.groupby('cell_id'):
    cell = cell.sort_values('cycle')
    ax.plot(cell['cycle'], cell['soh'] * 100, label=cell_id, alpha=0.85)
ax.axhline(80, color='red', linestyle='--', label='End-of-life (80%)')
ax.set_xlabel('Cycle'); ax.set_ylabel('SOH (%)')
ax.set_title('Capacity fade per cell')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

*Your observation here:* Look closely. Are there visible 'bumps' where SOH temporarily recovers? (Hint: there are. They're the well-known capacity-recovery effect — when a Li-ion cell rests, some capacity comes back temporarily before continuing to fade.) Is the slope of fade constant, or is there a 'knee' where it steepens? Where, roughly?

In [ ]:
# Distributions of the raw features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['voltage_mean', 'temperature_mean', 'discharge_duration_s']):
    for cell_id, cell in df.groupby('cell_id'):
        ax.hist(cell[col], bins=20, alpha=0.5, label=cell_id)
    ax.set_title(col); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# How does each feature correlate with SOH?
corr = df[['voltage_mean', 'current_mean', 'temperature_mean',
           'charge_duration_s', 'discharge_duration_s', 'soh']].corr()['soh'].sort_values()
corr

*Your observation here:* Which two features are most correlated (positively or negatively) with SOH? Does the sign make physical sense? (For example: a healthy cell should sustain its voltage longer under load — does the data agree?)

In [ ]:
# Trajectory plot: how do voltage and temperature evolve over cycles?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['voltage_mean', 'temperature_mean']):
    for cell_id, cell in df.groupby('cell_id'):
        cell = cell.sort_values('cycle')
        ax.plot(cell['cycle'], cell[col], label=cell_id, alpha=0.85)
    ax.set_xlabel('Cycle'); ax.set_ylabel(col)
    ax.set_title(f'{col} over the cell\'s lifetime')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

*Your observation here:* What direction does mean voltage drift over the cell's life? What about temperature? Tie this back to electrochemistry — what's happening physically?

These observations directly motivate the physics-derived features in `src/features.py` (Group C of the ablation):
- `voltage_decline` captures the voltage drift you (hopefully) just noticed.
- `temp_rise_vs_baseline` captures the temperature drift.
- `charge_to_discharge_ratio` captures internal-resistance growth indirectly.